# **Filter Response Normalization for Network-in-Network CIFAR-10 Classifier**

## **Libraries**

In [1]:
import torch
import torch.nn as nn
import os
import time
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import Subset
from torchvision import datasets
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image

In [2]:
class FilterResponseNormalization(nn.Module):
    def __init__(self, num_features, eps=1e-6):
        super(FilterResponseNormalization, self).__init__()
        
        self.register_parameter('beta', 
                                torch.nn.Parameter(
                                        torch.empty([1, num_features, 1, 1]).normal_()))
    
        self.register_parameter('gamma', 
                                torch.nn.Parameter(
                                        torch.empty([1, num_features, 1, 1]).normal_()))
        
        self.register_parameter('tau', 
                                torch.nn.Parameter(
                                        torch.empty([1, num_features, 1, 1]).normal_()))
        
        self.eps = torch.Tensor([eps])

    def forward(self, x):
        n, c, h, w = x.size()
        
        self.eps = self.eps.to(self.tau.device)

        nu2 = torch.mean(x.pow(2), (2, 3), keepdims=True)
        x = x * torch.rsqrt(nu2 + torch.abs(self.eps))
        return torch.max(self.gamma*x + self.beta, self.tau)

In [3]:
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True

## **Settings**

In [4]:
# Hyperparameters
RANDOM_SEED = 1
LEARNING_RATE = 0.00005
BATCH_SIZE = 256
NUM_EPOCHS = 10

# Architecture
NUM_CLASSES = 10

# Other
DEVICE = "cuda:0"
GRAYSCALE = False

## **CIFAR-10 Dataset**

In [5]:
train_indices = torch.arange(0, 49000)
valid_indices = torch.arange(49000, 50000)

train_and_valid = datasets.CIFAR10(root='data', 
                                   train=True, 
                                   transform=transforms.ToTensor(),
                                   download=True)

train_dataset = Subset(train_and_valid, train_indices)
valid_dataset = Subset(train_and_valid, valid_indices)

test_dataset = datasets.CIFAR10(root='data', 
                                train=False, 
                                transform=transforms.ToTensor())

100%|██████████| 170M/170M [00:02<00:00, 58.9MB/s] 


In [6]:
train_loader = DataLoader(dataset=train_dataset, 
                          batch_size=BATCH_SIZE,
                          num_workers=8,
                          shuffle=True)

valid_loader = DataLoader(dataset=valid_dataset, 
                          batch_size=BATCH_SIZE,
                          num_workers=8,
                          shuffle=False)

test_loader = DataLoader(dataset=test_dataset, 
                         batch_size=BATCH_SIZE,
                         num_workers=8,
                         shuffle=False)

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [7]:
for images, labels in train_loader:  
    print('Image batch dimensions:', images.shape)
    print('Image label dimensions:', labels.shape)
    break

for images, labels in test_loader:  
    print('Image batch dimensions:', images.shape)
    print('Image label dimensions:', labels.shape)
    break
    
for images, labels in valid_loader:  
    print('Image batch dimensions:', images.shape)
    print('Image label dimensions:', labels.shape)
    break

Image batch dimensions: torch.Size([256, 3, 32, 32])
Image label dimensions: torch.Size([256])
Image batch dimensions: torch.Size([256, 3, 32, 32])
Image label dimensions: torch.Size([256])
Image batch dimensions: torch.Size([256, 3, 32, 32])
Image label dimensions: torch.Size([256])


## **Filter Response Normalization**

In [8]:
class NiN(nn.Module):
    def __init__(self, num_classes):
        super(NiN, self).__init__()
        self.num_classes = num_classes
        self.classifier = nn.Sequential(
                nn.Conv2d(3, 192, kernel_size=5, stride=1, padding=2, bias=False),
                FilterResponseNormalization(192),
                #nn.ReLU(inplace=True),
                nn.Conv2d(192, 160, kernel_size=1, stride=1, padding=0, bias=False),
                FilterResponseNormalization(160),
                #nn.ReLU(inplace=True),
                nn.Conv2d(160,  96, kernel_size=1, stride=1, padding=0, bias=False),
                FilterResponseNormalization(96),
                #nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
                nn.Dropout(0.5),

                nn.Conv2d(96, 192, kernel_size=5, stride=1, padding=2, bias=False),
                FilterResponseNormalization(192),
                #nn.ReLU(inplace=True),
                nn.Conv2d(192, 192, kernel_size=1, stride=1, padding=0, bias=False),
                FilterResponseNormalization(192),
                #nn.ReLU(inplace=True),
                nn.Conv2d(192, 192, kernel_size=1, stride=1, padding=0, bias=False),
                FilterResponseNormalization(192),
                #nn.ReLU(inplace=True),
                nn.AvgPool2d(kernel_size=3, stride=2, padding=1),
                nn.Dropout(0.5),

                nn.Conv2d(192, 192, kernel_size=3, stride=1, padding=1, bias=False),
                FilterResponseNormalization(192),
                #nn.ReLU(inplace=True),
                nn.Conv2d(192, 192, kernel_size=1, stride=1, padding=0, bias=False),
                FilterResponseNormalization(192),
                #nn.ReLU(inplace=True),
                nn.Conv2d(192,  10, kernel_size=1, stride=1, padding=0),
                nn.ReLU(inplace=True),
                nn.AvgPool2d(kernel_size=8, stride=1, padding=0),
            )

    def forward(self, x):
        x = self.classifier(x)
        logits = x.view(x.size(0), self.num_classes)
        probas = torch.softmax(logits, dim=1)
        return logits, probas

In [9]:
torch.manual_seed(RANDOM_SEED)

model = NiN(NUM_CLASSES)
model.to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)  

In [10]:
def compute_accuracy(model, data_loader, device):
    correct_pred, num_examples = 0, 0
    for i, (features, targets) in enumerate(data_loader):
            
        features = features.to(device)
        targets = targets.to(device)

        logits, probas = model(features)
        _, predicted_labels = torch.max(probas, 1)
        num_examples += targets.size(0)
        correct_pred += (predicted_labels == targets).sum()
    return correct_pred.float()/num_examples * 100

In [11]:
start_time = time.time()
for epoch in range(NUM_EPOCHS):
    
    model.train()
    
    for batch_idx, (features, targets) in enumerate(train_loader):
    
        ### PREPARE MINIBATCH
        features = features.to(DEVICE)
        targets = targets.to(DEVICE)
            
        ### FORWARD AND BACK PROP
        logits, probas = model(features)
        cost = F.cross_entropy(logits, targets)
        optimizer.zero_grad()
        
        cost.backward()
        
        ### UPDATE MODEL PARAMETERS
        optimizer.step()
        
        ### LOGGING
        if not batch_idx % 120:
            print (f'Epoch: {epoch+1:03d}/{NUM_EPOCHS:03d} | '
                   f'Batch {batch_idx:03d}/{len(train_loader):03d} |' 
                   f' Cost: {cost:.4f}')

    with torch.set_grad_enabled(False):
        train_acc = compute_accuracy(model, train_loader, device=DEVICE)
        valid_acc = compute_accuracy(model, valid_loader, device=DEVICE)
        print(f'Epoch: {epoch+1:03d}/{NUM_EPOCHS:03d} Train Acc.: {train_acc:.2f}%'
              f' | Validation Acc.: {valid_acc:.2f}%')
        
    elapsed = (time.time() - start_time)/60
    print(f'Time elapsed: {elapsed:.2f} min')
  
elapsed = (time.time() - start_time)/60
print(f'Total Training Time: {elapsed:.2f} min')

Epoch: 001/010 | Batch 000/192 | Cost: 2.3029
Epoch: 001/010 | Batch 120/192 | Cost: 2.2052
Epoch: 001/010 Train Acc.: 23.13% | Validation Acc.: 26.30%
Time elapsed: 0.54 min
Epoch: 002/010 | Batch 000/192 | Cost: 2.1365
Epoch: 002/010 | Batch 120/192 | Cost: 2.0440
Epoch: 002/010 Train Acc.: 23.64% | Validation Acc.: 26.40%
Time elapsed: 1.06 min
Epoch: 003/010 | Batch 000/192 | Cost: 2.0690
Epoch: 003/010 | Batch 120/192 | Cost: 1.9951
Epoch: 003/010 Train Acc.: 28.04% | Validation Acc.: 30.40%
Time elapsed: 1.57 min
Epoch: 004/010 | Batch 000/192 | Cost: 1.9524
Epoch: 004/010 | Batch 120/192 | Cost: 1.8912
Epoch: 004/010 Train Acc.: 31.39% | Validation Acc.: 34.40%
Time elapsed: 2.09 min
Epoch: 005/010 | Batch 000/192 | Cost: 1.8684
Epoch: 005/010 | Batch 120/192 | Cost: 1.8213
Epoch: 005/010 Train Acc.: 33.32% | Validation Acc.: 34.70%
Time elapsed: 2.61 min
Epoch: 006/010 | Batch 000/192 | Cost: 1.8313
Epoch: 006/010 | Batch 120/192 | Cost: 1.7601
Epoch: 006/010 Train Acc.: 36.69%

## **Batch Normalization (for comparison)**

In [12]:
class NiN(nn.Module):
    def __init__(self, num_classes):
        super(NiN, self).__init__()
        self.num_classes = num_classes
        self.classifier = nn.Sequential(
                nn.Conv2d(3, 192, kernel_size=5, stride=1, padding=2, bias=False),
                nn.BatchNorm2d(192),
                nn.ReLU(inplace=True),
                nn.Conv2d(192, 160, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(160),
                nn.ReLU(inplace=True),
                nn.Conv2d(160,  96, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(96),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
                nn.Dropout(0.5),

                nn.Conv2d(96, 192, kernel_size=5, stride=1, padding=2, bias=False),
                nn.BatchNorm2d(192),
                nn.ReLU(inplace=True),
                nn.Conv2d(192, 192, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(192),
                nn.ReLU(inplace=True),
                nn.Conv2d(192, 192, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(192),
                nn.ReLU(inplace=True),
                nn.AvgPool2d(kernel_size=3, stride=2, padding=1),
                nn.Dropout(0.5),

                nn.Conv2d(192, 192, kernel_size=3, stride=1, padding=1, bias=False),
                nn.BatchNorm2d(192),
                nn.ReLU(inplace=True),
                nn.Conv2d(192, 192, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(192),
                nn.ReLU(inplace=True),
                nn.Conv2d(192,  10, kernel_size=1, stride=1, padding=0),
                nn.ReLU(inplace=True),
                nn.AvgPool2d(kernel_size=8, stride=1, padding=0),
                )

    def forward(self, x):
        x = self.classifier(x)
        logits = x.view(x.size(0), self.num_classes)
        probas = torch.softmax(logits, dim=1)
        return logits, probas

In [ ]:
start_time = time.time()
for epoch in range(NUM_EPOCHS):
    
    model.train()
    
    for batch_idx, (features, targets) in enumerate(train_loader):
    
        ### PREPARE MINIBATCH
        features = features.to(DEVICE)
        targets = targets.to(DEVICE)
            
        ### FORWARD AND BACK PROP
        logits, probas = model(features)
        cost = F.cross_entropy(logits, targets)
        optimizer.zero_grad()
        
        cost.backward()
        
        ### UPDATE MODEL PARAMETERS
        optimizer.step()
        
        ### LOGGING
        if not batch_idx % 120:
            print (f'Epoch: {epoch+1:03d}/{NUM_EPOCHS:03d} | '
                   f'Batch {batch_idx:03d}/{len(train_loader):03d} |' 
                   f' Cost: {cost:.4f}')

    with torch.set_grad_enabled(False):
        train_acc = compute_accuracy(model, train_loader, device=DEVICE)
        valid_acc = compute_accuracy(model, valid_loader, device=DEVICE)
        print(f'Epoch: {epoch+1:03d}/{NUM_EPOCHS:03d} Train Acc.: {train_acc:.2f}%'
              f' | Validation Acc.: {valid_acc:.2f}%')
        
    elapsed = (time.time() - start_time)/60
    print(f'Time elapsed: {elapsed:.2f} min')
  
elapsed = (time.time() - start_time)/60
print(f'Total Training Time: {elapsed:.2f} min')

Epoch: 001/010 | Batch 000/192 | Cost: 1.6494
Epoch: 001/010 | Batch 120/192 | Cost: 1.6053
Epoch: 001/010 Train Acc.: 42.52% | Validation Acc.: 42.20%
Time elapsed: 0.52 min
Epoch: 002/010 | Batch 000/192 | Cost: 1.6361
Epoch: 002/010 | Batch 120/192 | Cost: 1.6241
Epoch: 002/010 Train Acc.: 44.24% | Validation Acc.: 43.60%
Time elapsed: 1.03 min
Epoch: 003/010 | Batch 000/192 | Cost: 1.5914
Epoch: 003/010 | Batch 120/192 | Cost: 1.4696
Epoch: 003/010 Train Acc.: 45.65% | Validation Acc.: 44.40%
Time elapsed: 1.55 min
Epoch: 004/010 | Batch 000/192 | Cost: 1.6190
Epoch: 004/010 | Batch 120/192 | Cost: 1.6212
Epoch: 004/010 Train Acc.: 46.34% | Validation Acc.: 46.30%
Time elapsed: 2.07 min
Epoch: 005/010 | Batch 000/192 | Cost: 1.5458
Epoch: 005/010 | Batch 120/192 | Cost: 1.5455
Epoch: 005/010 Train Acc.: 45.89% | Validation Acc.: 44.90%
Time elapsed: 2.59 min
Epoch: 006/010 | Batch 000/192 | Cost: 1.4815
Epoch: 006/010 | Batch 120/192 | Cost: 1.5977
Epoch: 006/010 Train Acc.: 49.05%